In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/activity_labels.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/README.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/features_info.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/features.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/subject_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/y_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/X_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/body_acc_y_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/total_acc_y_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/total_acc_z_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/body_acc_z_test.txt
/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals/body_gyr

# Loading the Dataset

In [2]:
import numpy as np

def load_raw_signals(folder_path, filenames): # a function to load the dataset
    signals = []
    for name in filenames:
        file_path = f"{folder_path}/{name}.txt" # makes the file path fro every file
        data = np.loadtxt(file_path) #Loads the data from the file into an Numpy array
        signals.append(data) # appends the list we created with the new readings
    
    return np.transpose(np.array(signals), (1, 2, 0)) # for our project it requires we have the array in (sample, step, feature) format

# file names we want to load
filenames = [
    'body_acc_x_train', 'body_acc_y_train', 'body_acc_z_train',
    'body_gyro_x_train', 'body_gyro_y_train', 'body_gyro_z_train'
]

#The absolut path for the folder in our dataset where the files are located
raw_data_path = '/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/Inertial Signals'
X_raw = load_raw_signals(raw_data_path, filenames)

print(f"Raw 6-axis data shape: {X_raw.shape}") #printing the shape

Raw 6-axis data shape: (7352, 128, 6)


# Normalizing the data

In [3]:
import numpy as np

def load_raw_signals(folder_path, filenames):
    signals = []
    for name in filenames:
        file_path = f"{folder_path}/{name}.txt"
        data = np.loadtxt(file_path)
        signals.append(data)
    
    return np.transpose(np.array(signals), (1, 2, 0))
filenames = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z'
]
filenames_train = [f + '_train' for f in filenames]
filenames_test = [f + '_test' for f in filenames]
raw_data_path_train = '/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/Inertial Signals'
raw_data_path_test =  '/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/Inertial Signals'

x_train_raw = load_raw_signals(raw_data_path_train, filenames_train)
X_test_raw = load_raw_signals(raw_data_path_test, filenames_test)


mean = np.mean(x_train_raw, axis=(0, 1))
std = np.std(x_train_raw, axis=(0, 1))

#normalizing 
x_train = (x_train_raw - mean) / (std + 1e-8)
X_test = (X_test_raw - mean) / (std + 1e-8) 

# loading labels
y_train = np.loadtxt('/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/train/y_train.txt', dtype=int) - 1
y_test = np.loadtxt('/kaggle/input/datasets/rayanalam1/rawhar/UCI HAR Dataset/test/y_test.txt', dtype=int) - 1

print(f"Normalized x_train shape: {x_train.shape}")
print(f"Normalized X_test shape: {X_test.shape}")

Normalized x_train shape: (7352, 128, 6)
Normalized X_test shape: (2947, 128, 6)


# Coding the 1D-CNN Architecture

In [4]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers, models
model = keras.Sequential(name="HAR_1DCNN") 
model.add(layers.Input(shape=(128, 6))) # loading an empty stack to which we'll add layers
model.add(layers.Conv1D(   #Now we're adding layers to this 
    filters=64,    #this arg defines how many local features we'll look for
    kernel_size=5, #this is essentially our window size which we slide over our 128 timesteps to extract features
    activation='relu', #rectified linear unit introduces non-linearity basically acting as a gate to decide which information is important enough to be passed to the next layer
    padding= 'same',   #This makes sure that our output is the same length as our input adds zeros so the kernel can center itself
    #input_shape=(128,6) #This defines the size of our input passed 
))
#Stacking another layer cause we want a deeper model 
model.add(layers.MaxPooling1D(pool_size=2)) 
model.add(layers.Dropout(rate=0.5)) 

model.add(layers.Conv1D(   
    filters=128,   
    kernel_size=3, 
    activation='relu', 
    padding= 'same',  
))
model.add(layers.MaxPooling1D(pool_size=2))
model.add(layers.Dropout(rate=0.5))
model.add(layers.Flatten())


model.add(layers.Dense(128, activation='relu')) #input neurons


model.add(layers.Dropout(0.5)) #turns off 50% of neurons mid training to prevent memorization


model.add(layers.Dense(6, activation='softmax')) #softmax is a function that calculates probabilities of each of 6 outcomes and chooses the highest probability option
model.compile(
    optimizer='adam',  # updates weights and bias
    loss='sparse_categorical_crossentropy', #our loss function
    metrics=['accuracy'] # what metrics we'll be seeing
)

model.summary() # summary

history = model.fit(    #actual training logic epochs means training cycles btw
    x_train, y_train,
    epochs=30,
    batch_size=16,
    validation_data=(X_test, y_test), #validates accuracy against test data
    shuffle=True,   # shuffling the data
)

2026-05-06 17:06:07.080014: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778087167.293612      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778087167.352302      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778087167.834991      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778087167.835042      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778087167.835045      23 computation_placer.cc:177] computation placer alr

Model: "HAR_1DCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 128, 64)        │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 64, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 551,878 (2.11 MB)

 Trainable params: 551,878 (2.11 MB)

...

Epoch 1/30


I0000 00:00:1778087200.563409      70 service.cc:152] XLA service 0x785180009b40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778087200.563469      70 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778087200.563475      70 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778087200.965994      70 cuda_dnn.cc:529] Loaded cuDNN version 91002


 48/460 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.1995 - loss: 1.8816

I0000 00:00:1778087204.592894      70 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


460/460 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.3584 - loss: 1.3807 - val_accuracy: 0.6810 - val_loss: 0.6609
Epoch 2/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6429 - loss: 0.6759 - val_accuracy: 0.7540 - val_loss: 0.5901
Epoch 3/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7183 - loss: 0.5751 - val_accuracy: 0.7727 - val_loss: 0.5617
Epoch 4/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7660 - loss: 0.5021 - val_accuracy: 0.8341 - val_loss: 0.4507
Epoch 5/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8100 - loss: 0.4364 - val_accuracy: 0.8711 - val_loss: 0.3698
Epoch 6/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8339 - loss: 0.3720 - val_accuracy: 0.8717 - val_loss: 0.4138
Epoch 7/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8524 - loss: 0.3541 - val_accuracy: 0.8901 - val_loss: 0.3241
Epoch 8/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8765 - loss: 0.3040 - val_accuracy: 0.8870 - va